# Video Beauty Pipeline
Run each cell top to bottom. That's it.

In [ ]:
# ── Cell 1: Clone repo and install dependencies ──────────────────
# Replace the URL with your own repo once you push it to GitHub.
!git clone https://github.com/YOUR_USERNAME/video_beauty.git
%cd video_beauty
!pip install -r requirements.txt -q

In [ ]:
# ── Cell 2: Download face landmark model ─────────────────────────
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task \
     -O face_landmarker.task
print('face_landmarker.task ready.')

In [ ]:
# ── Cell 3: Download dataset ──────────────────────────────────────
# The dataset lives on Kaggle. You need a kaggle.json API token.
# Upload it when prompted, then run this cell.
from google.colab import files
print('Upload your kaggle.json API key:')
uploaded = files.upload()   # select kaggle.json from your machine

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kagglehub
path = kagglehub.dataset_download('pranavchandane/scut-fbp5500-v2-facial-beauty-scores')
print(f'Dataset downloaded to: {path}')

In [ ]:
# ── Cell 4: Verify environment detection ─────────────────────────
from config import print_env_info, make_output_dirs
make_output_dirs()
print_env_info()

In [ ]:
# ── Cell 5: Prepare data (align faces, create splits) ────────────
# This is the slow step — runs once, results cached to disk.
from data import prepare_data, make_dataloaders

train_df, val_df, test_df = prepare_data()
train_loader, val_loader, test_loader = make_dataloaders(train_df, val_df, test_df)

In [ ]:
# ── Cell 6: Train ─────────────────────────────────────────────────
# Auto-resumes from checkpoint if the session disconnected.
# Enable GPU: Runtime → Change runtime type → T4 GPU
from training import train

model = train(train_loader, val_loader)

In [ ]:
# ── Cell 7: Evaluate on test set ─────────────────────────────────
from training import evaluate

results = evaluate(model, test_loader)
print(f"Test MAE: {results['mae']:.4f}  |  Pearson r: {results['pearson_r']:.4f}")

In [ ]:
# ── Cell 7b: Grad-CAM feature importance ─────────────────────────
from model import GradCAM

sample_img, _ = next(iter(test_loader))
sample_img = sample_img[:1].to('cuda').requires_grad_(True)

cam     = GradCAM(model)
heatmap = cam.generate(sample_img)
print("Region importance:", cam.region_scores(heatmap))

overlay = cam.overlay(sample_img, heatmap)
cv2.imwrite(f'{EXPORT_DIR}/gradcam.png', overlay)

In [ ]:
# ── Cell 8: Export to TFLite (for Android) ───────────────────────
from export import export_pipeline

tflite_path = export_pipeline(model, val_loader)

In [ ]:
# ── Cell 9: Process a video ───────────────────────────────────────
# Upload your video first: Files panel (left sidebar) → Upload
from pathlib import Path
from config import EXPORT_DIR, SCORE_THRESHOLD, FRAME_SKIP
from video import process_video, print_summary, plot_score_timeline

INPUT_VIDEO   = '/content/your_video.mp4'   # <-- change this
THRESHOLD     = SCORE_THRESHOLD             # change here to override
FRAME_SKIP    = FRAME_SKIP

stem = Path(INPUT_VIDEO).stem
frame_scores, fps = process_video(
    model            = model,
    input_path       = INPUT_VIDEO,
    output_annotated = f'{EXPORT_DIR}/{stem}_annotated.mp4',
    output_cleaned   = f'{EXPORT_DIR}/{stem}_cleaned.mp4',
    threshold        = THRESHOLD,
    frame_skip       = FRAME_SKIP,
    filter_policy    = 'any',   # 'any' | 'all' | 'mean'
    bg_mode          = 'first_frame',
)

print_summary(frame_scores, fps)
plot_score_timeline(frame_scores, fps)

In [ ]:
# ── Cell 10: Download outputs ─────────────────────────────────────
from google.colab import files
import glob, os
from config import EXPORT_DIR

for f in glob.glob(f'{EXPORT_DIR}/*.mp4') + \
         glob.glob(f'{EXPORT_DIR}/*.tflite') + \
         glob.glob(f'{EXPORT_DIR}/*.png'):
    print(f'Downloading: {os.path.basename(f)}')
    files.download(f)